# SpideyNet — MNIST CNN Lab

**Goal:** build, inspect, train, evaluate, and improve a convolutional neural network while understanding *why* each design choice matters.

This notebook uses PyTorch + torchvision and is designed for Google Colab (GPU) or a local Jupyter environment.

### Experiments
1. Verify the data pipeline and tensor shapes.
2. Inspect convolution/pooling feature-map dimensions and parameter count.
3. Train a compact CNN baseline with correct loss accounting.
4. Evaluate accuracy, per-class performance, and a confusion matrix.
5. Inspect mistakes and visualize learned convolution filters.
6. Run controlled experiments: augmentation, model capacity, and regularization.

**Important:** keep the test set for final evaluation. Use validation data for choosing hyperparameters. For a rigorous comparison, do not repeatedly tune against the test set.

## 1. Imports, reproducibility, and device

In [ ]:
import random
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms

from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Data pipeline

MNIST images are grayscale, so the input shape is **`[batch, 1, 28, 28]`**.

We normalize using the commonly used MNIST training-set statistics. Random rotation is applied **only to training data**. Applying random augmentation to test data would make evaluation stochastic and unfair.

We split 5,000 images from the original training set for validation. The official 10,000-image test set is kept separate.

In [ ]:
DATA_DIR = Path("./data")  # Colab persists this in the runtime; change if desired
BATCH_SIZE = 128
NUM_WORKERS = 2 if device.type == "cuda" else 0
PIN_MEMORY = device.type == "cuda"

MNIST_MEAN = (0.1307,)
MNIST_STD = (0.3081,)

train_transform = transforms.Compose([
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(MNIST_MEAN, MNIST_STD),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MNIST_MEAN, MNIST_STD),
])

# Two views of the same original training split:
# augmented view for training, deterministic view for validation.
train_aug_full = datasets.MNIST(
    root=DATA_DIR, train=True, download=True, transform=train_transform
)
train_eval_full = datasets.MNIST(
    root=DATA_DIR, train=True, download=True, transform=eval_transform
)
test_data = datasets.MNIST(
    root=DATA_DIR, train=False, download=True, transform=eval_transform
)

generator = torch.Generator().manual_seed(SEED)
train_indices, val_indices = random_split(
    range(len(train_aug_full)),
    [55_000, 5_000],
    generator=generator
)
train_indices = list(train_indices)
val_indices = list(val_indices)

train_data = Subset(train_aug_full, train_indices)
val_data = Subset(train_eval_full, val_indices)

train_loader = DataLoader(
    train_data, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)
val_loader = DataLoader(
    val_data, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)
test_loader = DataLoader(
    test_data, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)

print(f"Train: {len(train_data):,} | Validation: {len(val_data):,} | Test: {len(test_data):,}")
print(f"Batches: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}")

### Inspect a batch

`ToTensor()` converts a grayscale PIL image into a float tensor with shape `[1, 28, 28]` and values in `[0, 1]`; normalization then shifts/scales those values. Undo normalization for display.

In [ ]:
images, labels = next(iter(train_loader))
print("Image batch:", images.shape)
print("Label batch:", labels.shape)
print("Image dtype:", images.dtype, "| label dtype:", labels.dtype)
print("Normalized pixel range:", (images.min().item(), images.max().item()))

def show_mnist_batch(images, labels, count=12):
    mean = torch.tensor(MNIST_MEAN).view(1, 1, 1, 1)
    std = torch.tensor(MNIST_STD).view(1, 1, 1, 1)
    display_images = images[:count].cpu() * std + mean

    fig, axes = plt.subplots(2, (count + 1) // 2, figsize=(12, 4))
    for ax, image, label in zip(axes.flat, display_images, labels[:count]):
        ax.imshow(image.squeeze(), cmap="gray")
        ax.set_title(f"Label: {label.item()}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_mnist_batch(images, labels)

## 3. CNN architecture — shape mathematics

For a convolution or pooling layer, each spatial output dimension is

\[
H_{out}=\left\lfloor\frac{H_{in}+2P-D(K-1)-1}{S}+1\right\rfloor
\]

where `K` = kernel size, `P` = padding, `S` = stride, and `D` = dilation.

Our network uses two downsampling stages. Convolutions with `kernel_size=3, padding=1, stride=1` preserve height and width. A `2×2` max pool halves them.

| Stage | Output shape (excluding batch) |
|---|---|
| Input | `1 × 28 × 28` |
| Conv block 1 | `32 × 28 × 28` |
| MaxPool | `32 × 14 × 14` |
| Conv block 2 | `64 × 14 × 14` |
| MaxPool | `64 × 7 × 7` |
| Adaptive average pool | `128 × 1 × 1` |
| Classifier | `10` logits |

**Why not 512 channels here?** MNIST is small and simple. A 512-channel stage can add substantial compute and parameters without guaranteeing better generalization. Start with a compact model, measure it, then scale deliberately.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, pool=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        ]
        if pool:
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class SpideyNet(nn.Module):
    def __init__(self, dropout=0.25):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1, 32, pool=True),   # 28 -> 14
            ConvBlock(32, 64, pool=True),  # 14 -> 7
            ConvBlock(64, 128, pool=False)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(p=dropout),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)  # raw logits; do not apply softmax before CrossEntropyLoss


model = SpideyNet(dropout=0.25).to(device)
print(model)

### Inspect every feature-map shape and count parameters

This is a useful debugging habit: verify the tensor dimensions rather than guessing them.

In [ ]:
def inspect_model(model, input_shape=(1, 1, 28, 28)):
    x = torch.zeros(input_shape, device=device)
    print("Input:", tuple(x.shape))
    model.eval()
    with torch.inference_mode():
        for name, layer in model.features.named_children():
            x = layer(x)
            print(f"{name:>3} {layer.__class__.__name__:>12}: {tuple(x.shape)}")
        x = model.classifier(x)
    print("Logits:", tuple(x.shape))

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total:,}")
    print(f"Trainable parameters: {trainable:,}")

inspect_model(model)
count_parameters(model)

## 4. Loss, optimizer, and training/evaluation functions

### Important fixes compared with the first notebook
- Iterate over the function argument `dataloader`, not a hard-coded `train_loader`.
- Accumulate `loss.item()` as a float; **do not wrap it in `int()`** or small losses become zero.
- Divide by the number of examples, weighted by batch size. This handles a smaller final batch correctly.
- `CrossEntropyLoss` expects raw logits and integer class labels (`torch.long`).
- `model.train()` enables training behavior; `model.eval()` disables dropout and uses BatchNorm running statistics.
- `torch.inference_mode()` prevents gradient tracking during evaluation.
- `optimizer` should be a `torch.optim.Optimizer`, not an `nn.Module`.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for X, y in dataloader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        batch_size = y.size(0)
        running_loss += loss.item() * batch_size
        predictions = logits.argmax(dim=1)
        correct += (predictions == y).sum().item()
        total += batch_size

    return running_loss / total, correct / total


@torch.inference_mode()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for X, y in dataloader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(X)
        loss = criterion(logits, y)

        batch_size = y.size(0)
        running_loss += loss.item() * batch_size
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += batch_size

    return running_loss / total, correct / total

## 5. Train with validation tracking

Start with 8–12 epochs. This model is small; inspect the curves rather than assuming that more epochs are always better. We save the best model according to validation accuracy.

In [ ]:
EPOCHS = 10
history = {
    "train_loss": [], "train_acc": [],
    "val_loss": [], "val_acc": []
}
best_val_acc = 0.0
best_model_path = Path("spideynet_mnist_best.pth")

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    val_loss, val_acc = evaluate(
        model, val_loader, criterion, device
    )

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)

    elapsed = time.time() - start_time
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {train_loss:.4f} acc {train_acc:.4%} | "
        f"val loss {val_loss:.4f} acc {val_acc:.4%} | "
        f"{elapsed:.1f}s"
    )

print(f"Best validation accuracy: {best_val_acc:.4%}")
print("Best weights saved to:", best_model_path)

## 6. Plot learning curves

Interpretation: if training loss keeps falling while validation loss rises, the model may be overfitting. If both losses remain high, the model may be underfitting or the optimization setup may need attention.

In [ ]:
epochs_axis = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs_axis, history["train_loss"], marker="o", label="Train")
axes[0].plot(epochs_axis, history["val_loss"], marker="o", label="Validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs_axis, history["train_acc"], marker="o", label="Train")
axes[1].plot(epochs_axis, history["val_acc"], marker="o", label="Validation")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 7. Final test-set evaluation

Load the checkpoint with the best validation accuracy, then evaluate the official test set once. Do not use test accuracy to repeatedly choose model settings.

In [ ]:
best_model = SpideyNet(dropout=0.25).to(device)
best_model.load_state_dict(torch.load(best_model_path, map_location=device))
best_model.eval()

test_loss, test_acc = evaluate(best_model, test_loader, criterion, device)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4%}")

### Confusion matrix and per-class report

In [ ]:
@torch.inference_mode()
def collect_predictions(model, dataloader, device):
    model.eval()
    all_targets, all_predictions = [], []
    for X, y in dataloader:
        logits = model(X.to(device))
        all_predictions.extend(logits.argmax(dim=1).cpu().tolist())
        all_targets.extend(y.tolist())
    return np.array(all_targets), np.array(all_predictions)

y_true, y_pred = collect_predictions(best_model, test_loader, device)
print(classification_report(y_true, y_pred, digits=4))

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(range(10)))
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("MNIST test confusion matrix")
plt.show()

## 8. Inspect the model's mistakes

Look at incorrect predictions. Ask: are the digits genuinely ambiguous, unusually written, or consistently confused with another class?

In [ ]:
wrong_indices = np.where(y_true != y_pred)[0]
print(f"Incorrect predictions: {len(wrong_indices)} / {len(y_true)}")

def show_mistakes(model, dataset, indices, count=12):
    count = min(count, len(indices))
    if count == 0:
        print("No mistakes found in this subset.")
        return

    fig, axes = plt.subplots(3, 4, figsize=(10, 8))
    for ax, idx in zip(axes.flat, indices[:count]):
        image, label = dataset[int(idx)]
        with torch.inference_mode():
            pred = model(image.unsqueeze(0).to(device)).argmax(1).item()
        image_to_show = image.squeeze().cpu() * MNIST_STD[0] + MNIST_MEAN[0]
        ax.imshow(image_to_show, cmap="gray")
        ax.set_title(f"True: {label} | Pred: {pred}")
        ax.axis("off")
    for ax in axes.flat[count:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_mistakes(best_model, test_data, wrong_indices)

## 9. Visualize first-layer filters

A first-layer filter is a small learnable kernel. The values are not manually programmed edge detectors: they are learned through gradients to reduce classification loss. Later layers combine earlier feature maps into more complex representations.

In [ ]:
first_conv = best_model.features[0].block[0]
filters = first_conv.weight.detach().cpu()  # [out_channels, in_channels, kh, kw]

fig, axes = plt.subplots(4, 8, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(filters[i, 0], cmap="gray")
    ax.set_title(f"Filter {i}")
    ax.axis("off")
plt.suptitle("Learned first-layer convolution kernels")
plt.tight_layout()
plt.show()

## 10. Deep-learning experiments — change one variable at a time

Do not change augmentation, architecture, optimizer, learning rate, and dropout simultaneously. That makes it impossible to know what caused a change.

Suggested experiments (record validation accuracy, training time, parameter count, and the train/validation gap):

| Experiment | Change | Question |
|---|---|---|
| A — baseline | Current settings | Establish a reference |
| B — no augmentation | Remove `RandomRotation` | Does rotation help generalization? |
| C — stronger augmentation | Try rotation 15° | Does more variation help or distort digits? |
| D — capacity | Compare channels 16/32/64 vs 32/64/128 | Does a larger model help? |
| E — regularization | Dropout 0.0 vs 0.25 vs 0.5 | Does dropout reduce the generalization gap? |
| F — optimizer | AdamW vs SGD + momentum | How does optimization affect convergence? |

### Exercise: calculate the receptive field

For a stack of layers, track:
- `r`: receptive-field size
- `j`: jump (distance between adjacent feature positions in input pixels)

Initialize `r=1, j=1`. For each layer with kernel `k` and stride `s`:
\[
r_{new}=r+(k-1)j, \qquad j_{new}=js
\]
Try calculating the receptive field at each block by hand, then compare it with the code below.

In [ ]:
def receptive_field(layers):
    receptive_field_size = 1
    jump = 1
    for name, kernel, stride in layers:
        receptive_field_size += (kernel - 1) * jump
        jump *= stride
        print(
            f"{name:>12}: receptive_field={receptive_field_size}, "
            f"jump={jump}"
        )

# Conv3, Pool2, Conv3, Pool2, Conv3
receptive_field([
    ("conv1", 3, 1),
    ("pool1", 2, 2),
    ("conv2", 3, 1),
    ("pool2", 2, 2),
    ("conv3", 3, 1),
])

## 11. Save and reload the trained model

Save the `state_dict`, not the whole Python model object. Recreate the same architecture before loading it.

In [ ]:
MODEL_PATH = Path("spideynet_mnist_final.pth")
torch.save(best_model.state_dict(), MODEL_PATH)

reloaded_model = SpideyNet(dropout=0.25).to(device)
reloaded_model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
reloaded_model.eval()

print("Saved and reloaded:", MODEL_PATH.resolve())

## 12. Knowledge check

Answer these before moving on:

1. Why does `CrossEntropyLoss` receive logits rather than softmax probabilities?
2. Why must `model.eval()` be called during evaluation when the model has BatchNorm and Dropout?
3. Why is `loss.item()` accumulated as a float instead of converted to `int`?
4. Why should random augmentation be applied to training data but not validation/test data?
5. What does max pooling do to spatial resolution? What information might it discard?
6. Why can a model with more channels perform worse on unseen data?
7. What does a confusion-matrix cell at row 4, column 9 mean?
8. How does increasing the receptive field change the visual context available to a neuron?

**Next level:** compare this CNN with a simple fully connected network, then study residual connections, learning-rate scheduling, mixed precision, saliency maps, and transfer learning on a more complex dataset.